# 01 — Exploratory Data Analysis
Load your dataset and explore it before moving on to preprocessing.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
DATA_PATH = '../data/raw/dataset.csv'
TARGET_COL = 'target'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Column types and general info
df.info()

In [ ]:
# Descriptive statistics
df.describe(include='all').T

In [ ]:
# Missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0]

if missing_df.empty:
    print('No missing values.')
else:
    print(missing_df)
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_df['pct'].plot(kind='bar', ax=ax, color='coral')
    ax.set_title('Missing values per column (%)')
    ax.set_ylabel('%')
    plt.tight_layout()
    plt.show()

In [ ]:
# Target distribution
fig, ax = plt.subplots(figsize=(8, 4))
if df[TARGET_COL].dtype == object or df[TARGET_COL].nunique() <= 20:
    df[TARGET_COL].value_counts().plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'Distribution of "{TARGET_COL}"')
else:
    df[TARGET_COL].hist(bins=30, ax=ax, color='steelblue')
    ax.set_title(f'Histogram of "{TARGET_COL}"')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix (numerical features)
num_df = df.select_dtypes(include='number')
if num_df.shape[1] > 1:
    corr = num_df.corr()
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Correlation matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution of numerical features
num_cols = num_df.drop(columns=[TARGET_COL], errors='ignore').columns.tolist()
if num_cols:
    n_cols = min(3, len(num_cols))
    n_rows = (len(num_cols) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.array(axes).flatten()
    for i, col in enumerate(num_cols):
        df[col].hist(bins=30, ax=axes[i], color='teal', alpha=0.7)
        axes[i].set_title(col)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
# Categorical features
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if TARGET_COL in cat_cols:
    cat_cols.remove(TARGET_COL)

for col in cat_cols[:6]:  # limit to 6 for readability
    print(f'\n{col} — {df[col].nunique()} unique values')
    print(df[col].value_counts().head(10))